<a href="https://colab.research.google.com/github/NatSy77/mlops_ai-projet6et8/blob/main/02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Initiez-vous au MLOps (partie 1/2)

In [1]:
# Monter google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys
import pandas as pd
import numpy as np

PROJECT_PATH = "/content/drive/MyDrive/Colab Notebooks/AI_projet_6&8"
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/AI_projet_6&8/credit_scoring_data/raw"

sys.path.append(PROJECT_PATH)

print("PROJECT_PATH:", PROJECT_PATH)
print("DATA_PATH:", DATA_PATH)
print("Fichiers dispo:", sorted(os.listdir(DATA_PATH)))

PROJECT_PATH: /content/drive/MyDrive/Colab Notebooks/AI_projet_6&8
DATA_PATH: /content/drive/MyDrive/Colab Notebooks/AI_projet_6&8/credit_scoring_data/raw
Fichiers dispo: ['HomeCredit_columns_description.csv', 'POS_CASH_balance.csv', 'application_test.csv', 'application_train.csv', 'bureau.csv', 'bureau_balance.csv', 'credit_card_balance.csv', 'installments_payments.csv', 'previous_application.csv', 'sample_submission.csv']


## Chargement des tables

In [3]:
app_train = pd.read_csv(f"{DATA_PATH}/application_train.csv")
app_test = pd.read_csv(f"{DATA_PATH}/application_test.csv")
bureau = pd.read_csv(f"{DATA_PATH}/bureau.csv")
bureau_balance = pd.read_csv(f"{DATA_PATH}/bureau_balance.csv")
previous_application = pd.read_csv(f"{DATA_PATH}/previous_application.csv")
installments_payments = pd.read_csv(f"{DATA_PATH}/installments_payments.csv")
pos_cash = pd.read_csv(f"{DATA_PATH}/POS_CASH_balance.csv")
credit_card = pd.read_csv(f"{DATA_PATH}/credit_card_balance.csv")

In [4]:
tables = {
    "app_train": app_train,
    "app_test": app_test,
    "bureau": bureau,
    "bureau_balance": bureau_balance,
    "previous_application": previous_application,
    "installments_payments": installments_payments,
    "pos_cash": pos_cash,
    "credit_card": credit_card,
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

app_train: (307511, 122)
app_test: (48744, 121)
bureau: (1716428, 17)
bureau_balance: (27299925, 3)
previous_application: (1670214, 37)
installments_payments: (13605401, 8)
pos_cash: (10001358, 8)
credit_card: (3840312, 23)


## Vérification des clés de jointure

In [5]:
print("application_train columns clés :", [c for c in app_train.columns if "SK_ID" in c])
print("bureau columns clés :", [c for c in bureau.columns if "SK_ID" in c])
print("bureau_balance columns clés :", [c for c in bureau_balance.columns if "SK_ID" in c])
print("previous_application columns clés :", [c for c in previous_application.columns if "SK_ID" in c])
print("installments_payments columns clés :", [c for c in installments_payments.columns if "SK_ID" in c])
print("pos_cash columns clés :", [c for c in pos_cash.columns if "SK_ID" in c])
print("credit_card columns clés :", [c for c in credit_card.columns if "SK_ID" in c])

application_train columns clés : ['SK_ID_CURR']
bureau columns clés : ['SK_ID_CURR', 'SK_ID_BUREAU']
bureau_balance columns clés : ['SK_ID_BUREAU']
previous_application columns clés : ['SK_ID_PREV', 'SK_ID_CURR']
installments_payments columns clés : ['SK_ID_PREV', 'SK_ID_CURR']
pos_cash columns clés : ['SK_ID_PREV', 'SK_ID_CURR']
credit_card columns clés : ['SK_ID_PREV', 'SK_ID_CURR']


## Étape 1

- résumer bureau_balance au niveau SK_ID_BUREAU
- fusionner ce résumé dans bureau
- résumer bureau au niveau client SK_ID_CURR
- merger dans app_train et app_test

### 1. bureau_balance puis bureau

In [6]:
print("Doublons app_train:", app_train.duplicated(subset=["SK_ID_CURR"]).sum())
print("Doublons app_test:", app_test.duplicated(subset=["SK_ID_CURR"]).sum())
print("Doublons bureau:", bureau.duplicated(subset=["SK_ID_BUREAU"]).sum())

Doublons app_train: 0
Doublons app_test: 0
Doublons bureau: 0


### 2. Explorer bureau_balance

In [7]:
# Explorer bureau_balance
bureau_balance.head()
bureau_balance["STATUS"].value_counts(dropna=False)

,count
STATUS,
C,13646993
0,7499507
X,5810482
1,242347
5,62406
2,23419
3,8924
4,5847


### 3. Agrégation de bureau_balance par SK_ID_BUREAU

In [8]:
# One-hot encoding de STATUS
bb_ohe = pd.get_dummies(bureau_balance, columns=["STATUS"], dummy_na=False)

# Agrégation par crédit bureau
bb_agg = bb_ohe.groupby("SK_ID_BUREAU").agg({
    "MONTHS_BALANCE": ["min", "max", "size"],
    **{col: ["mean"] for col in bb_ohe.columns if col.startswith("STATUS_")}
})

# Aplatir les colonnes
bb_agg.columns = [
    "BB_" + "_".join(col).upper()
    for col in bb_agg.columns.to_flat_index()
]

bb_agg = bb_agg.reset_index()

print(bb_agg.shape)
bb_agg.head()

(817395, 12)


,SK_ID_BUREAU,BB_MONTHS_BALANCE_MIN,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_SIZE,BB_STATUS_0_MEAN,BB_STATUS_1_MEAN,BB_STATUS_2_MEAN,BB_STATUS_3_MEAN,BB_STATUS_4_MEAN,BB_STATUS_5_MEAN,BB_STATUS_C_MEAN,BB_STATUS_X_MEAN
0,5001709,-96,0,97,0.000000,0.0,0.0,0.0,0.0,0.0,0.886598,0.113402
1,5001710,-82,0,83,0.060241,0.0,0.0,0.0,0.0,0.0,0.578313,0.361446
2,5001711,-3,0,4,0.750000,0.0,0.0,0.0,0.0,0.0,0.000000,0.250000
3,5001712,-18,0,19,0.526316,0.0,0.0,0.0,0.0,0.0,0.473684,0.000000
4,5001713,-21,0,22,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,1.000000


### 4. Fusionner bb_agg dans bureau

In [9]:
bureau_full = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")

print("bureau avant merge:", bureau.shape)
print("bureau après merge:", bureau_full.shape)
bureau_full.head()

bureau avant merge: (1716428, 17)
bureau après merge: (1716428, 28)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,BB_MONTHS_BALANCE_MAX,BB_MONTHS_BALANCE_SIZE,BB_STATUS_0_MEAN,BB_STATUS_1_MEAN,BB_STATUS_2_MEAN,BB_STATUS_3_MEAN,BB_STATUS_4_MEAN,BB_STATUS_5_MEAN,BB_STATUS_C_MEAN,BB_STATUS_X_MEAN
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5. Préparer bureau avant agrégation

Il y a des variables catégorielles dans bureau, donc on les encode avant agrégation.

In [10]:
bureau_full = pd.get_dummies(bureau_full, columns=["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"], dummy_na=False)

print(bureau_full.shape)

(1716428, 48)


### 6. Agrégation de bureau_full par SK_ID_CURR

In [11]:
num_agg_dict = {
    "DAYS_CREDIT": ["min", "max", "mean"],
    "CREDIT_DAY_OVERDUE": ["max", "mean"],
    "DAYS_CREDIT_ENDDATE": ["min", "max", "mean"],
    "DAYS_ENDDATE_FACT": ["min", "max", "mean"],
    "AMT_CREDIT_MAX_OVERDUE": ["max", "mean"],
    "CNT_CREDIT_PROLONG": ["sum", "mean"],
    "AMT_CREDIT_SUM": ["max", "mean", "sum"],
    "AMT_CREDIT_SUM_DEBT": ["max", "mean", "sum"],
    "AMT_CREDIT_SUM_LIMIT": ["max", "mean"],
    "AMT_CREDIT_SUM_OVERDUE": ["max", "mean", "sum"],
    "DAYS_CREDIT_UPDATE": ["min", "max", "mean"],
    "AMT_ANNUITY": ["max", "mean"],
}

# Ajouter les colonnes BB_ si elles existent
bb_cols = [c for c in bureau_full.columns if c.startswith("BB_")]
for col in bb_cols:
    num_agg_dict[col] = ["mean"]

# Colonnes one-hot du bureau
cat_cols = [
    c for c in bureau_full.columns
    if c.startswith("CREDIT_ACTIVE_")
    or c.startswith("CREDIT_CURRENCY_")
    or c.startswith("CREDIT_TYPE_")
]

bureau_agg = bureau_full.groupby("SK_ID_CURR").agg(
    {**num_agg_dict, **{col: ["mean"] for col in cat_cols}}
)

bureau_agg.columns = [
    "BURO_" + "_".join(col).upper()
    for col in bureau_agg.columns.to_flat_index()
]

bureau_agg = bureau_agg.reset_index()

print(bureau_agg.shape)
bureau_agg.head()

(305811, 66)


,SK_ID_CURR,BURO_DAYS_CREDIT_MIN,BURO_DAYS_CREDIT_MAX,BURO_DAYS_CREDIT_MEAN,BURO_CREDIT_DAY_OVERDUE_MAX,BURO_CREDIT_DAY_OVERDUE_MEAN,BURO_DAYS_CREDIT_ENDDATE_MIN,BURO_DAYS_CREDIT_ENDDATE_MAX,BURO_DAYS_CREDIT_ENDDATE_MEAN,BURO_DAYS_ENDDATE_FACT_MIN,...,BURO_CREDIT_TYPE_INTERBANK CREDIT_MEAN,BURO_CREDIT_TYPE_LOAN FOR BUSINESS DEVELOPMENT_MEAN,BURO_CREDIT_TYPE_LOAN FOR PURCHASE OF SHARES (MARGIN LENDING)_MEAN,BURO_CREDIT_TYPE_LOAN FOR THE PURCHASE OF EQUIPMENT_MEAN,BURO_CREDIT_TYPE_LOAN FOR WORKING CAPITAL REPLENISHMENT_MEAN,BURO_CREDIT_TYPE_MICROLOAN_MEAN,BURO_CREDIT_TYPE_MOBILE OPERATOR LOAN_MEAN,BURO_CREDIT_TYPE_MORTGAGE_MEAN,BURO_CREDIT_TYPE_REAL ESTATE LOAN_MEAN,BURO_CREDIT_TYPE_UNKNOWN TYPE OF LOAN_MEAN
0,100001,-1572,-49,-735.000000,0,0.0,-1329.0,1778.0,82.428571,-1328.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,100002,-1437,-103,-874.000000,0,0.0,-1072.0,780.0,-349.000000,-1185.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,100003,-2586,-606,-1400.750000,0,0.0,-2434.0,1216.0,-544.500000,-2131.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,100004,-1326,-408,-867.000000,0,0.0,-595.0,-382.0,-488.500000,-683.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,100005,-373,-62,-190.666667,0,0.0,-128.0,1324.0,439.333333,-123.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 7. Merge dans app_train et app_test

In [12]:
train_fe = app_train.merge(bureau_agg, on="SK_ID_CURR", how="left")
test_fe = app_test.merge(bureau_agg, on="SK_ID_CURR", how="left")

print("app_train avant:", app_train.shape)
print("train_fe après bureau:", train_fe.shape)

print("app_test avant:", app_test.shape)
print("test_fe après bureau:", test_fe.shape)

app_train avant: (307511, 122)
train_fe après bureau: (307511, 187)
app_test avant: (48744, 121)
test_fe après bureau: (48744, 186)


### 8. Vérification importante après merge

In [13]:
print("Doublons train_fe:", train_fe.duplicated(subset=["SK_ID_CURR"]).sum())
print("Doublons test_fe:", test_fe.duplicated(subset=["SK_ID_CURR"]).sum())

Doublons train_fe: 0
Doublons test_fe: 0


#### Recap :
La table bureau_balance a d’abord été agrégée au niveau SK_ID_BUREAU, puis fusionnée à la table bureau.
La table enrichie a ensuite été agrégée au niveau SK_ID_CURR afin d’obtenir des variables historiques consolidées par client, puis fusionnée à application_train et application_test.
Après fusion avec application_train et application_test, le nombre de lignes est resté inchangé, ce qui confirme l’absence de duplication induite par les jointures.
Cette étape a permis d’enrichir les données avec 65 nouvelles variables décrivant l’historique de crédit externe des clients.

## Étape 2
previous_application

Bloc utile qui donne l’historique des demandes précédentes du client.

Objectif:

- encoder les colonnes catégorielles
- agréger par SK_ID_CURR
- merger dans train_fe et test_fe

### 1. Vérification rapide

In [14]:
print(previous_application.shape)
print(previous_application["SK_ID_CURR"].nunique())
print(previous_application.duplicated(subset=["SK_ID_PREV"]).sum())
previous_application.head()

(1670214, 37)
338857
0


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


### 2. Repérer les colonnes catégorielles

In [15]:
prev_cat_cols = previous_application.select_dtypes(include=["object"]).columns.tolist()
print("Colonnes catégorielles :", prev_cat_cols)
print("Nb colonnes catégorielles :", len(prev_cat_cols))

Colonnes catégorielles : ['NAME_CONTRACT_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'NAME_SELLER_INDUSTRY', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION']
Nb colonnes catégorielles : 16


### 3. One-hot encoding

In [16]:
prev_ohe = pd.get_dummies(previous_application, columns=prev_cat_cols, dummy_na=False)
print(prev_ohe.shape)

(1670214, 164)


### 4. Agrégation par client

In [17]:
prev_num_agg = {
    "AMT_ANNUITY": ["min", "max", "mean"],
    "AMT_APPLICATION": ["min", "max", "mean"],
    "AMT_CREDIT": ["min", "max", "mean"],
    "AMT_DOWN_PAYMENT": ["min", "max", "mean"],
    "AMT_GOODS_PRICE": ["min", "max", "mean"],
    "HOUR_APPR_PROCESS_START": ["min", "max", "mean"],
    "RATE_DOWN_PAYMENT": ["min", "max", "mean"],
    "DAYS_DECISION": ["min", "max", "mean"],
    "CNT_PAYMENT": ["mean", "sum"],
}

prev_cat_encoded_cols = [c for c in prev_ohe.columns if c not in previous_application.columns and c != "SK_ID_CURR"]

prev_agg = prev_ohe.groupby("SK_ID_CURR").agg(
    {**prev_num_agg, **{col: ["mean"] for col in prev_cat_encoded_cols}}
)

prev_agg.columns = [
    "PREV_" + "_".join(col).upper()
    for col in prev_agg.columns.to_flat_index()
]

prev_agg = prev_agg.reset_index()

print(prev_agg.shape)
prev_agg.head()

(338857, 170)


,SK_ID_CURR,PREV_AMT_ANNUITY_MIN,PREV_AMT_ANNUITY_MAX,PREV_AMT_ANNUITY_MEAN,PREV_AMT_APPLICATION_MIN,PREV_AMT_APPLICATION_MAX,PREV_AMT_APPLICATION_MEAN,PREV_AMT_CREDIT_MIN,PREV_AMT_CREDIT_MAX,PREV_AMT_CREDIT_MEAN,...,PREV_PRODUCT_COMBINATION_CASH X-SELL: LOW_MEAN,PREV_PRODUCT_COMBINATION_CASH X-SELL: MIDDLE_MEAN,PREV_PRODUCT_COMBINATION_POS HOUSEHOLD WITH INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS HOUSEHOLD WITHOUT INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS INDUSTRY WITH INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS INDUSTRY WITHOUT INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS MOBILE WITH INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS MOBILE WITHOUT INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS OTHER WITH INTEREST_MEAN,PREV_PRODUCT_COMBINATION_POS OTHERS WITHOUT INTEREST_MEAN
0,100001,3951.000,3951.000,3951.000,24835.5,24835.5,24835.50,23787.0,23787.0,23787.00,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,1.0,0.0,0.0,0.0
1,100002,9251.775,9251.775,9251.775,179055.0,179055.0,179055.00,179055.0,179055.0,179055.00,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,1.0,0.0
2,100003,6737.310,98356.995,56553.990,68809.5,900000.0,435436.50,68053.5,1035882.0,484191.00,...,0.333333,0.0,0.333333,0.0,0.333333,0.0,0.0,0.0,0.0,0.0
3,100004,5357.250,5357.250,5357.250,24282.0,24282.0,24282.00,20106.0,20106.0,20106.00,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.0,1.0,0.0,0.0
4,100005,4813.200,4813.200,4813.200,0.0,44617.5,22308.75,0.0,40153.5,20076.75,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.5,0.0,0.0,0.0


### 5. Merge dans train_fe et test_fe

In [18]:
train_fe = train_fe.merge(prev_agg, on="SK_ID_CURR", how="left")
test_fe = test_fe.merge(prev_agg, on="SK_ID_CURR", how="left")

print("train_fe après previous_application:", train_fe.shape)
print("test_fe après previous_application:", test_fe.shape)

print("Doublons train_fe:", train_fe.duplicated(subset=["SK_ID_CURR"]).sum())
print("Doublons test_fe:", test_fe.duplicated(subset=["SK_ID_CURR"]).sum())

train_fe après previous_application: (307511, 356)
test_fe après previous_application: (48744, 355)
Doublons train_fe: 0
Doublons test_fe: 0


#### Recap
La table previous_application a été agrégée au niveau client afin de résumer l’historique des demandes de crédit antérieures.
Les variables numériques ont été résumées par des statistiques simples (min, max, mean, sum) et les variables catégorielles ont été encodées puis agrégées par fréquence moyenne.

## Étape 3 : installments_payments

Cette table permet de capter le comportement de remboursement.

On va créer des variables comme :

- écart entre montant payé et montant dû
retard de paiement
- fréquence des paiements en retard

### 1. Features de base

In [19]:
inst = installments_payments.copy()

inst["PAYMENT_PERC"] = inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"]
inst["PAYMENT_DIFF"] = inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]
inst["DPD"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]
inst["DBD"] = inst["DAYS_INSTALMENT"] - inst["DAYS_ENTRY_PAYMENT"]

inst["DPD"] = inst["DPD"].apply(lambda x: x if x > 0 else 0)
inst["DBD"] = inst["DBD"].apply(lambda x: x if x > 0 else 0)

inst.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT,PAYMENT_PERC,PAYMENT_DIFF,DPD,DBD
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360,1.000000,0.000,0.0,7.0
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525,1.000000,0.000,0.0,0.0
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000,1.000000,0.000,0.0,0.0
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130,1.000000,0.000,0.0,8.0
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585,0.997942,4.455,17.0,0.0


### 2. Agrégation par client

In [20]:
inst_agg = inst.groupby("SK_ID_CURR").agg({
    "NUM_INSTALMENT_VERSION": ["nunique"],
    "DPD": ["max", "mean", "sum"],
    "DBD": ["max", "mean", "sum"],
    "PAYMENT_PERC": ["max", "mean", "var"],
    "PAYMENT_DIFF": ["max", "mean", "var"],
    "AMT_INSTALMENT": ["max", "mean", "sum"],
    "AMT_PAYMENT": ["min", "max", "mean", "sum"],
    "DAYS_ENTRY_PAYMENT": ["max", "mean"],
})

inst_agg.columns = [
    "INSTAL_" + "_".join(col).upper()
    for col in inst_agg.columns.to_flat_index()
]

inst_agg = inst_agg.reset_index()

print(inst_agg.shape)
inst_agg.head()

(339587, 23)


,SK_ID_CURR,INSTAL_NUM_INSTALMENT_VERSION_NUNIQUE,INSTAL_DPD_MAX,INSTAL_DPD_MEAN,INSTAL_DPD_SUM,INSTAL_DBD_MAX,INSTAL_DBD_MEAN,INSTAL_DBD_SUM,INSTAL_PAYMENT_PERC_MAX,INSTAL_PAYMENT_PERC_MEAN,...,INSTAL_PAYMENT_DIFF_VAR,INSTAL_AMT_INSTALMENT_MAX,INSTAL_AMT_INSTALMENT_MEAN,INSTAL_AMT_INSTALMENT_SUM,INSTAL_AMT_PAYMENT_MIN,INSTAL_AMT_PAYMENT_MAX,INSTAL_AMT_PAYMENT_MEAN,INSTAL_AMT_PAYMENT_SUM,INSTAL_DAYS_ENTRY_PAYMENT_MAX,INSTAL_DAYS_ENTRY_PAYMENT_MEAN
0,100001,2,11.0,1.571429,11.0,36.0,8.857143,62.0,1.0,1.0,...,0.0,17397.900,5885.132143,41195.925,3951.000,17397.900,5885.132143,41195.925,-1628.0,-2195.000000
1,100002,2,0.0,0.000000,0.0,31.0,20.421053,388.0,1.0,1.0,...,0.0,53093.745,11559.247105,219625.695,9251.775,53093.745,11559.247105,219625.695,-49.0,-315.421053
2,100003,2,0.0,0.000000,0.0,14.0,7.160000,179.0,1.0,1.0,...,0.0,560835.360,64754.586000,1618864.650,6662.970,560835.360,64754.586000,1618864.650,-544.0,-1385.320000
3,100004,2,0.0,0.000000,0.0,11.0,7.666667,23.0,1.0,1.0,...,0.0,10573.965,7096.155000,21288.465,5357.250,10573.965,7096.155000,21288.465,-727.0,-761.666667
4,100005,2,1.0,0.111111,1.0,37.0,23.666667,213.0,1.0,1.0,...,0.0,17656.245,6240.205000,56161.845,4813.200,17656.245,6240.205000,56161.845,-470.0,-609.555556


3. Merge

In [21]:
train_fe = train_fe.merge(inst_agg, on="SK_ID_CURR", how="left")
test_fe = test_fe.merge(inst_agg, on="SK_ID_CURR", how="left")

print("train_fe après installments:", train_fe.shape)
print("test_fe après installments:", test_fe.shape)

print("Doublons train_fe:", train_fe.duplicated(subset=["SK_ID_CURR"]).sum())
print("Doublons test_fe:", test_fe.duplicated(subset=["SK_ID_CURR"]).sum())

train_fe après installments: (307511, 378)
test_fe après installments: (48744, 377)
Doublons train_fe: 0
Doublons test_fe: 0


In [22]:
inst.replace([np.inf, -np.inf], np.nan, inplace=True)

#### Recap
La table installments_payments a été utilisée pour construire des variables comportementales liées au remboursement, notamment les retards de paiement (DPD), les avances de paiement (DBD), le ratio entre montant payé et montant dû, ainsi que les écarts de paiement. Ces variables sont particulièrement pertinentes dans un contexte de scoring crédit.

## Étape 4 : Nettoyage final + dataset final

### 1. Gestion des valeurs infinies

In [22]:
train_fe.replace([np.inf, -np.inf], np.nan, inplace=True)
test_fe.replace([np.inf, -np.inf], np.nan, inplace=True)

### 2. Vérifier les NaN

In [23]:
missing_train = train_fe.isnull().mean().sort_values(ascending=False)
missing_train.head(10)

,0
BURO_AMT_ANNUITY_MAX,0.739817
BURO_AMT_ANNUITY_MEAN,0.739817
BURO_BB_STATUS_X_MEAN_MEAN,0.700073
BURO_BB_STATUS_C_MEAN_MEAN,0.700073
BURO_BB_MONTHS_BALANCE_SIZE_MEAN,0.700073
BURO_BB_MONTHS_BALANCE_MAX_MEAN,0.700073
BURO_BB_MONTHS_BALANCE_MIN_MEAN,0.700073
BURO_BB_STATUS_0_MEAN_MEAN,0.700073
BURO_BB_STATUS_5_MEAN_MEAN,0.700073
BURO_BB_STATUS_3_MEAN_MEAN,0.700073


Une proportion importante de clients ne possède pas d’historique de crédit externe, ce qui est cohérent avec le contexte métier (clients peu ou pas bancarisés).

### 3. Supprimer colonnes trop vides

In [24]:
threshold = 0.8

cols_to_drop = missing_train[missing_train > threshold].index.tolist()

train_fe.drop(columns=cols_to_drop, inplace=True)
test_fe.drop(columns=cols_to_drop, inplace=True)

print("Nb colonnes supprimées:", len(cols_to_drop))

Nb colonnes supprimées: 0


### 4. Séparer target

In [25]:
y = train_fe["TARGET"]
X = train_fe.drop(columns=["TARGET"])

### 5. Align train / test

In [26]:
# Colonnes communes
common_cols = list(set(X.columns).intersection(set(test_fe.columns)))

# Réduction mémoire (important)
X = X[common_cols].copy()
test_fe = test_fe[common_cols].copy()

print("X shape:", X.shape)
print("test shape:", test_fe.shape)

X shape: (307511, 377)
test shape: (48744, 377)


### Optimisation RAM

In [27]:
def reduce_memory(df):
    for col in df.columns:
        if df[col].dtype == "float64":
            df[col] = df[col].astype("float32")
        elif df[col].dtype == "int64":
            df[col] = df[col].astype("int32")
    return df

X = reduce_memory(X)
test_fe = reduce_memory(test_fe)

### 6. Sauvegarde dataset final

In [28]:
SAVE_PATH = "/content/drive/MyDrive/Colab Notebooks/AI_projet_6&8/credit_scoring_data/processed/"

os.makedirs(SAVE_PATH, exist_ok=True)

X.to_csv(SAVE_PATH + "X_train.csv", index=False)
y.to_csv(SAVE_PATH + "y_train.csv", index=False)
test_fe.to_csv(SAVE_PATH + "X_test.csv", index=False)

#### Recap
Les données ont été nettoyées en supprimant les colonnes présentant plus de 80 % de valeurs manquantes.
Les jeux d’entraînement et de test ont ensuite été alignés pour garantir la cohérence des features.
Le dataset final est prêt pour l’entraînement du modèle.